# dcgan-normal-init-002 — ex1: apply normal(0, 0.02) init to Conv/ConvTranspose via model.apply

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dcgan-normal-init-002`. Running the final beacon cell reports progress against the `GAN: DCGAN normal init 0.02` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: DCGAN normal init 0.02` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dcgan-normal-init-002`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dcgan-normal-init-002"
DD_SUBTOPIC = "GAN: DCGAN normal init 0.02"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## DCGAN normal init (mean=0, std=0.02) — quick refresher

The Radford et al. DCGAN paper specifies a fixed init for every Conv / ConvTranspose weight: `N(0, 0.02)`. The canonical implementation walks the model and applies the init to matching layers:

```python
def init_dcgan(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, mean=0.0, std=0.02)

model.apply(init_dcgan)
```

**Why std=0.02, not the PyTorch default.** The default for `Conv2d` is Kaiming-uniform — fine for ReLU classifiers but too wide for adversarial training. A small std (0.02) keeps activations bounded early in training so the discriminator doesn't immediately saturate and starve the generator of gradient.

**`model.apply(fn)` recurses.** Walks every submodule (depth-first) and calls `fn(submodule)`. The function should be a no-op for layers it doesn't recognize — hence the `isinstance` guard.

### Exercise 1 — apply normal(0, 0.02) init to Conv/ConvTranspose via model.apply

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `nn.init.normal_(m.weight, 0.0, 0.02)` to every Conv2d and ConvTranspose2d submodule of a model using `model.apply(init_fn)`, leaving other layer types untouched.
> Keywords: dcgan, init, model.apply, normal-init
> ```

**KCs targeted:** `dcgan-conv-normal-init`, `model-apply-recursion`

Implement `ex1_apply_dcgan_init(model)`. The Radford et al. DCGAN convolutional weight initializer:

1. Define a local function `init_fn(m)` that:
   - Checks `isinstance(m, (nn.Conv2d, nn.ConvTranspose2d))`.
   - If so, calls `nn.init.normal_(m.weight, mean=0.0, std=0.02)` (in-place).
   - Does NOTHING for other module types (BatchNorm, Linear, Sequential, the model itself).
2. Call `model.apply(init_fn)` to walk every submodule.
3. Return `model` (mutated in place, but returning is conventional).

Input: `model` — `nn.Module`, may contain Conv2d, ConvTranspose2d, BatchNorm2d, Linear submodules (mix any).
Output: same `model` with Conv / ConvT weights resampled from `N(0, 0.02)`. Other layers untouched.

The visualization runs your init on a small DCGAN-style model and renders before/after histograms of Conv2d weight values so you can verify the distribution change.

In [ ]:
def ex1_apply_dcgan_init(model: nn.Module) -> nn.Module:
    """Apply N(0, 0.02) to Conv/ConvTranspose weights via model.apply."""
    raise NotImplementedError()


def _test_ex1():
    import torch.nn as nn

    # Build a mixed model — has Conv, ConvT, BN, Linear.
    model = nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.BatchNorm2d(16),
        nn.ConvTranspose2d(16, 8, 4, stride=2, padding=1),
        nn.Conv2d(8, 4, 1),
        nn.Flatten(),
        nn.Linear(64, 10),
    )

    # Snapshot BN gamma + Linear weight BEFORE init — must remain unchanged.
    bn_w_before = model[1].weight.detach().clone()
    lin_w_before = model[5].weight.detach().clone()

    out = ex1_apply_dcgan_init(model)
    assert out is model, 'must return the same model (mutated in place)'

    # Conv layers must now have weight std ~ 0.02 (sample-size loose tolerance).
    conv_layers = [m for m in model.modules() if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d))]
    assert len(conv_layers) == 3, f'expected 3 conv layers, found {len(conv_layers)}'
    for layer in conv_layers:
        s = layer.weight.std().item()
        mean = layer.weight.mean().item()
        assert abs(s - 0.02) < 0.01, f'expected std ~0.02, got {s:.5f} for {type(layer).__name__}'
        assert abs(mean) < 0.01, f'expected mean ~0, got {mean:.5f}'

    # BatchNorm weight + Linear weight must be UNCHANGED.
    assert t.equal(model[1].weight, bn_w_before), 'BatchNorm weight must be untouched'
    assert t.equal(model[5].weight, lin_w_before), 'Linear weight must be untouched'

    # Stress test — nested Sequential, still walks recursively.
    nested = nn.Sequential(
        nn.Sequential(nn.Conv2d(3, 8, 3), nn.BatchNorm2d(8)),
        nn.Sequential(nn.ConvTranspose2d(8, 16, 4)),
    )
    ex1_apply_dcgan_init(nested)
    for m in nested.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            assert abs(m.weight.std().item() - 0.02) < 0.015, 'nested conv weight not initialized'

    # --- Visualization: weight histogram before vs after on a fresh model ---
    viz_model = nn.Sequential(
        nn.Conv2d(3, 64, 4, stride=2, padding=1),
        nn.Conv2d(64, 128, 4, stride=2, padding=1),
        nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
    )
    before_vals = t.cat([m.weight.detach().flatten() for m in viz_model.modules() if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d))])
    ex1_apply_dcgan_init(viz_model)
    after_vals = t.cat([m.weight.detach().flatten() for m in viz_model.modules() if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d))])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    ax1.hist(before_vals.numpy(), bins=60, color='gray', edgecolor='black')
    ax1.set_title(f'before init — std={before_vals.std().item():.4f}')
    ax1.set_xlabel('weight value'); ax1.set_ylabel('count')
    ax2.hist(after_vals.numpy(), bins=60, color='steelblue', edgecolor='black')
    ax2.set_title(f'after DCGAN init — std={after_vals.std().item():.4f}')
    ax2.set_xlabel('weight value')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_apply_dcgan_init(model: nn.Module) -> nn.Module:
    def init_fn(m):
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    model.apply(init_fn)
    return model
```

**`model.apply` does the recursion for you.** No need to write `for m in model.modules()` — `apply` walks every submodule and calls `init_fn(m)`. The `isinstance` guard inside keeps the function a no-op for layers we don't want to touch.

**Why std=0.02 specifically.** Radford et al. picked this by experiment — small enough to prevent immediate D saturation, large enough to break symmetry. Don't substitute Kaiming / Xavier in a DCGAN; the gradient balance breaks.

**Bias is untouched.** DCGAN convolutions often have `bias=False` (because BatchNorm follows). When `bias=True`, leaving it at PyTorch's default zero-init is fine.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()